# Setup

In [1]:
import os
if os.path.exists('reviews_classifier'):
    !rm -rf reviews_classifier
!git clone https://github.com/Sirius-Siru/reviews_classifier.git

import sys
if '/content/reviews_classifier' not in sys.path:
    sys.path.append('/content/reviews_classifier')

!pip install -q kaggle

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

if os.path.isdir('data') and os.path.exists('data/testData.tsv'):
    print("✨ Dữ liệu đã sẵn sàng, không cần tải lại.")
else:
    print("🚀 Đang tải dữ liệu...")
    !kaggle competitions download -c word2vec-nlp-tutorial
    !unzip -o word2vec-nlp-tutorial.zip -d data
    !unzip -o data/labeledTrainData.tsv.zip -d data
    !unzip -o data/unlabeledTrainData.tsv.zip -d data
    !unzip -o data/testData.tsv.zip -d data

    !rm -f word2vec-nlp-tutorial.zip
    !rm -f data/labeledTrainData.tsv.zip
    !rm -f data/unlabeledTrainData.tsv.zip
    !rm -f data/testData.tsv.zip

Cloning into 'reviews_classifier'...


remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 46 (delta 14), reused 40 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 7.82 KiB | 7.82 MiB/s, done.
Resolving deltas: 100% (14/14), done.
✨ Dữ liệu đã sẵn sàng, không cần tải lại.


In [2]:
import pandas as pd
import numpy as np
import csv
import matplotlib.pyplot as plt
import gc

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import src.preprocessing as pcs

# Load data

In [3]:
labeledTrain = pd.read_csv('data/labeledTrainData.tsv', sep='\t', quoting=csv.QUOTE_NONE)
unlabeledTrain = pd.read_csv('data/unlabeledTrainData.tsv', sep='\t', quoting=csv.QUOTE_NONE)
test = pd.read_csv('data/testData.tsv', sep='\t', quoting=csv.QUOTE_NONE)
print(labeledTrain.info())
print(unlabeledTrain.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         25000 non-null  object
 1   sentiment  25000 non-null  int64 
 2   review     25000 non-null  object
dtypes: int64(1), object(2)
memory usage: 586.1+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      50000 non-null  object
 1   review  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB
None


# Preprocessing

In [4]:
labeledTrain['review'] = labeledTrain['review'].apply(pcs.clean)
unlabeledTrain['review'] = unlabeledTrain['review'].apply(pcs.clean)
test['review'] = test['review'].apply(pcs.clean)

custom_stopwords = [
    "i", "you", "he", "she", "it", "we", "they",
    "me", "him", "her", "us", "them",
    "my", "your", "his", "their",
    "this", "that", "these", "those",
    "of", 'for', 'with', 'which',
    'what', 'when', 'where', 'how',
    'who'
]

vectorizer = TfidfVectorizer(
    stop_words=custom_stopwords,
    max_df = 0.7,
    min_df = 5,
    ngram_range = (1, 3)
)

ul_aug = unlabeledTrain['review'].apply(pcs.aug)
l_aug = labeledTrain['review'].apply(pcs.aug)

ul_txt = np.concatenate((unlabeledTrain['review'].values, ul_aug), axis = 0)
l_txt = np.concatenate((labeledTrain['review'].values, l_aug), axis = 0)

y = np.concatenate((labeledTrain['sentiment'].values, labeledTrain['sentiment'].values), axis = 0)

ul_txt = vectorizer.fit_transform(ul_txt)
l_txt = vectorizer.fit_transform(l_txt)
test_txt = vectorizer.transform(test['review'])

del ul_aug, l_aug
gc.collect()


2

# Training

In [5]:
X_train, X_val, y_train, y_val = train_test_split(l_txt, y, test_size=0.2, 
                                                  random_state=42)
print(type(X_train))
print(type(y_train))

<class 'scipy.sparse._csr.csr_matrix'>
<class 'numpy.ndarray'>


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
import numpy as np
from lightgbm import LGBMClassifier

#Logistic
lr = LogisticRegression(max_iter=2000, C = 4, n_jobs=-1)

lr.fit(X_train, y_train)
y_pred = lr.predict(X_val)
accu1 = accuracy_score(y_val, y_pred) * 100

#SVC
svc = LinearSVC(C = 1)
svc.fit(X_train, y_train)
y_pred = svc.predict(X_val)
accu2 = accuracy_score(y_val, y_pred) * 100

#Naive Bayes + Logistic
y_train = np.array(y_train)
def pr(x, y_i, y):
    return (x[y == y_i].sum(0) + 1) / ((y == y_i).sum() + 1)

r = np.log(pr(X_train, 1, y_train) / pr(X_train, 0, y_train))

X_nb = X_train.multiply(r)
X_val_nb = X_val.multiply(r)

lr = LogisticRegression(max_iter=2000, C = 4, n_jobs=-1)
lr.fit(X_nb, y_train)

y_pred = lr.predict(X_val_nb)
accu3 = accuracy_score(y_val, y_pred) * 100

# #Lightgbm
# lgbm = LGBMClassifier(
#     n_estimators=500,
#     learning_rate=0.05,
#     num_leaves=31
# )
# lgbm.fit(X_train, y_train)

# y_pred = lgbm.predict(X_val)
# accu4 = accuracy_score(y_val, y_pred) * 100

print(f"{accu1:.2f}")
print(f"{accu2:.2f}")
print(f"{accu3:.2f}")
# print(f"{accu4:.2f}")

95.04
96.57
91.96


# CNN

In [6]:
from tensorflow.keras import layers, models

cnn = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Reshape((X_train.shape[1], 1)),
    
    # Block 1
    layers.Conv1D(64, kernel_size=3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),
    
    # Block 2
    layers.Conv1D(128, kernel_size=3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),
    
    # Block 3
    layers.Conv1D(256, kernel_size=3, padding='same', activation='relu'),
    layers.GlobalMaxPooling1D(),
    
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [7]:
history = cnn.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_val, y_val)
)

Epoch 1/5


: 

: 

: 

In [ ]:
# svc.fit(l_txt, y)
# submission = svc.predict(test_txt)
# submission = pd.concat([test['id'], pd.DataFrame(submission)], axis=1)
# submission.columns = ['id', 'sentiment']
# submission['id'] = submission['id'].str.strip('"')
# submission.info()
# print(submission.head(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         25000 non-null  object
 1   sentiment  25000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 390.8+ KB
         id  sentiment
0  12311_10          1
1    8348_2          0
2    5828_4          1
3    7186_2          1
4   12128_7          1


In [ ]:
# submission.to_csv('submission.csv', index=False)
# !kaggle competitions submit -c word2vec-nlp-tutorial -f submission.csv  -m "baseline"

100% 227k/227k [00:00<00:00, 1.11MB/s]
Successfully submitted to Bag of Words Meets Bags of Popcorn